### Classification of FR abstracts as big data or not.

Uses TFIDF as vectorization and initial labeled dataset of 1200 abstracts.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import time
import math
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import classification_report, confusion_matrix

pd.set_option('display.max_columns', 50)

#### Ingest Data - labeled set and full corpus

In [ ]:
# read in initial training dataset - 1200 labeled abstracts
df_lab = pd.read_csv("../../../../data/prd/Digital_abstract_labelled/AUG-2022-labelled_abstracts.csv")
print(df_lab.shape)
df_lab.head()

In [ ]:
df_lab = df_lab[['PROJECT_ID','ABSTRACT','Is it related to Big-Data','label']]
df_lab.head()

In [ ]:
#data frame with all of the abstracts
df = pd.read_pickle("../../../../data/prd/Paper/FR_meta_and_final_tokens_23DEC21.pkl")

# Reset the df index and create a variable index to link with PROJECT_ID
df.reset_index(inplace = True, drop = True)
print(df.shape)
df.head()

In [ ]:
df = df[['PROJECT_ID','ABSTRACT']]
df.head()

In [ ]:
# merge with the labelled data using PROJECT_ID.
df['PROJECT_ID'] = pd.to_numeric(df['PROJECT_ID'])
df_merge = df.merge(df_lab[['PROJECT_ID','Is it related to Big-Data','label']], how='left', on='PROJECT_ID')
len(df_merge)

In [ ]:
df_merge.head()

In [ ]:
# Save project ID of labelled data
project_id_lab = list(df_lab['PROJECT_ID'])

# Get the index of labelled abstract
subset_df = df_merge.loc[df_merge['PROJECT_ID'].isin(project_id_lab),['PROJECT_ID','label']]

In [ ]:
subset_df.head()

In [ ]:
# build training and testing sets from the labeled data 
# sklearn to find list of indices for training and test set

X_train_idx, X_test_idx, y_train_tfidf, y_test_tfidf = train_test_split(subset_df.index, 
                                                                        subset_df['label'].values, 
                                                                        test_size=0.20, random_state=0, 
                                                                        stratify=subset_df['label'].values)


### 1.1 TFIDF Vectorization

In [ ]:
# Compute the TFIDF matrix on the full data
t1 = time.time()
v = TfidfVectorizer()
tfidf = v.fit_transform(df['ABSTRACT'])
t2 = time.time()

print(t2-t1)

In [ ]:
tfidf

In [ ]:
# use training and testing set ids to create feature sets

X_train_tfidf = tfidf[X_train_idx,:]
X_test_tfidf = tfidf[X_test_idx,:]

## 1.2. KNN classifier

In [ ]:
# number of neighbors
k = int(math.sqrt(len(y_train_tfidf)))
print('Number of neighbors:', k)

In [ ]:
# Run KNN

classifierKNN = KNeighborsClassifier(n_neighbors= k)  # njobs? don't need

# cross-validation with 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(classifierKNN, X_train_tfidf, y_train_tfidf, scoring='accuracy', cv=cv)
print(scores)
print("Accuracy: %0.2f.   Standard Deviation: %0.2f" % (scores.mean(), scores.std()))

# train model after cross-validation
classifierKNN.fit(X_train_tfidf, y_train_tfidf)

# find model accuracy
classifierKNN.score(X_test_tfidf,y_test_tfidf)

In [ ]:
# Predict the classification
y_pred_tfidf = classifierKNN.predict(X_test_tfidf)

In [ ]:
# Classifier performance
confusion = confusion_matrix(y_test_tfidf, y_pred_tfidf)
print(confusion)
print(classification_report(y_test_tfidf, y_pred_tfidf))

In [ ]:
## Display the visualization of the Confusion Matrix.

sns.set(font_scale=1.4)
ax = sns.heatmap(confusion, annot=True, fmt="", cmap='Blues', xticklabels=["Big Data", "Not Big Data"], 
            annot_kws={"fontsize":20})
plt.title('KNN Confusion Matrix\n TFIDF \n') 
plt.xlabel('\nPredicted Values')
plt.ylabel('Actual Values') 
ax.set_yticklabels(['Big Data', 'Not Big Data'], va='center', rotation = 90) #position=(0,0.28))

plt.savefig('figures/tfidf_KNN.png', bbox_inches='tight')
plt.show()

In [ ]:
# Predict the classification for the whole data: use the whole TFIDF
t1 = time.time()
y_tfidf = classifierKNN.predict(tfidf)
t2 = time.time()

print(t2-t1)

In [ ]:
# The TFIDF keep the order from index. Just add the prediction as new variable
df_merge['Prediction_KNN'] = y_tfidf
df_merge.head()

## 1.3. SVM

In [ ]:
# We run SVM with default parameters provides by Sklearn. We train the model

classifier_svm = svm.SVC()

# cross-validation with 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(classifier_svm, X_train_tfidf, y_train_tfidf, scoring='accuracy', cv=cv)
print(scores)
print("Accuracy: %0.2f.   Standard Deviation: %0.2f" % (scores.mean(), scores.std()))

# train model after cross-validation
classifier_svm.fit(X_train_tfidf, y_train_tfidf)

# find model accuracy
classifier_svm.score(X_test_tfidf,y_test_tfidf)

In [ ]:
# Predict the classification
y_pred_tfidf = classifier_svm.predict(X_test_tfidf)

In [ ]:
# Classifier performance
confusion = confusion_matrix(y_test_tfidf, y_pred_tfidf)
print(confusion)
print(classification_report(y_test_tfidf, y_pred_tfidf))

In [ ]:
## Display the visualization of the Confusion Matrix.

sns.set(font_scale=1.4)
ax = sns.heatmap(confusion, annot=True, fmt="", cmap='Blues', xticklabels=["Big Data", "Not Big Data"], 
            annot_kws={"fontsize":20})
plt.title('SVM Confusion Matrix\n TFIDF \n') 
plt.xlabel('\nPredicted Values')
plt.ylabel('Actual Values') 
ax.set_yticklabels(['Big Data', 'Not Big Data'], va='center', rotation = 90) #position=(0,0.28))

plt.savefig('figures/tfidf_SVM.png', bbox_inches='tight')
plt.show()

In [ ]:
# Predict the classification for the whole data: use the whole TFIDF
t1 = time.time()
y_tfidf = classifier_svm.predict(tfidf)
t2 = time.time()

print(t2-t1)

In [ ]:
# The TFIDF keep the order from index. Just add the prediction as new variable
df_merge['Prediction_SVM'] = y_tfidf
df_merge.head()

## 1.4. Decision Tree

In [ ]:
# Use default parameters provides by SKLearn and train the model
classifier_DT = DecisionTreeClassifier(random_state = 0)

# cross-validation with 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(classifier_DT, X_train_tfidf, y_train_tfidf, scoring='accuracy', cv=cv)
print(scores)
print("Accuracy: %0.2f.   Standard Deviation: %0.2f" % (scores.mean(), scores.std()))

# train model after cross-validation
classifier_DT.fit(X_train_tfidf, y_train_tfidf)

# find model accuracy
classifier_DT.score(X_test_tfidf,y_test_tfidf)

In [ ]:
# Predict the classification
y_pred_tfidf = classifier_DT.predict(X_test_tfidf)

In [ ]:
# Classifier performance
confusion = confusion_matrix(y_test_tfidf, y_pred_tfidf)
print(confusion)
print(classification_report(y_test_tfidf, y_pred_tfidf))

In [ ]:
## Display the visualization of the Confusion Matrix.

sns.set(font_scale=1.4)
ax = sns.heatmap(confusion, annot=True, fmt="", cmap='Blues', xticklabels=["Big Data", "Not Big Data"], 
            annot_kws={"fontsize":20})
plt.title('Decision Tree Confusion Matrix\n TFIDF \n') 
plt.xlabel('\nPredicted Values')
plt.ylabel('Actual Values') 
ax.set_yticklabels(['Big Data', 'Not Big Data'], va='center', rotation = 90) #position=(0,0.28))

plt.savefig('figures/tfidf_DT.png', bbox_inches='tight')
plt.show()

In [ ]:
# Predict the classification for the whole data: use the whole TFIDF
t1 = time.time()
y_tfidf = classifier_DT.predict(tfidf)
t2 = time.time()

print(t2-t1)

In [ ]:
# The TFIDF keep the order from index. Just add the prediction as new variable
df_merge['Prediction_DT'] = y_tfidf
df_merge.head()

## 1.5. Random Forest

In [ ]:
# Use default parameters provides by SKLearn and train the model
classifier_RF = RandomForestClassifier(random_state=0)  # use njobs?

# cross-validation with 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(classifier_RF, X_train_tfidf, y_train_tfidf, scoring='accuracy', cv=cv)
print(scores)
print("Accuracy: %0.2f.   Standard Deviation: %0.2f" % (scores.mean(), scores.std()))

# train model after cross-validation
classifier_RF.fit(X_train_tfidf, y_train_tfidf)

# find model accuracy
classifier_RF.score(X_test_tfidf,y_test_tfidf)

In [ ]:
# Predict the classification
y_pred_tfidf = classifier_RF.predict(X_test_tfidf)

In [ ]:
# Classifier performance
confusion = confusion_matrix(y_test_tfidf, y_pred_tfidf)
print(confusion)
print(classification_report(y_test_tfidf, y_pred_tfidf))

In [ ]:
## Display the visualization of the Confusion Matrix.

sns.set(font_scale=1.4)
ax = sns.heatmap(confusion, annot=True, fmt="", cmap='Blues', xticklabels=["Big Data", "Not Big Data"], 
            annot_kws={"fontsize":20})
plt.title('Random Forest Confusion Matrix\n TFIDF \n') 
plt.xlabel('\nPredicted Values')
plt.ylabel('Actual Values') 
ax.set_yticklabels(['Big Data', 'Not Big Data'], va='center', rotation = 90) #position=(0,0.28))

plt.savefig('figures/tfidf_RF.png', bbox_inches='tight')
plt.show()

In [ ]:
# Predict the classification for the whole data: use the whole TFIDF
t1 = time.time()
y_tfidf = classifier_RF.predict(tfidf)
t2 = time.time()

print(t2-t1)

In [ ]:
# The TFIDF keep the order from index. Just add the prediction as new variable
df_merge['Prediction_RF'] = y_tfidf
df_merge.head()

## 1.6. Neural network classifier

In [ ]:
# Use default parameters provides by SKLearn and train the model
classifier_nn = MLPClassifier(random_state=0, verbose=True)

# cross-validation with 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(classifier_nn, X_train_tfidf, y_train_tfidf, scoring='accuracy', cv=cv)
print(scores)
print("Accuracy: %0.2f.   Standard Deviation: %0.2f" % (scores.mean(), scores.std()))

# train model after cross-validation
t1 = time.time()
classifier_nn.fit(X_train_tfidf, y_train_tfidf)
t2 = time.time()

print(t2-t1)

# find model accuracy
classifier_nn.score(X_test_tfidf,y_test_tfidf)

In [ ]:
# Predict the classification
y_pred_tfidf = classifier_nn.predict(X_test_tfidf)

In [ ]:
# Classifier performance
confusion = confusion_matrix(y_test_tfidf, y_pred_tfidf)
print(confusion)
print(classification_report(y_test_tfidf, y_pred_tfidf))

In [ ]:
## Display the visualization of the Confusion Matrix.

sns.set(font_scale=1.4)
ax = sns.heatmap(confusion, annot=True, fmt="", cmap='Blues', xticklabels=["Big Data", "Not Big Data"], 
            annot_kws={"fontsize":20})
plt.title('Neural Network Confusion Matrix\n TFIDF \n') 
plt.xlabel('\nPredicted Values')
plt.ylabel('Actual Values') 
ax.set_yticklabels(['Big Data', 'Not Big Data'], va='center', rotation = 90) #position=(0,0.28))

plt.savefig('figures/tfidf_NN.png', bbox_inches='tight')
plt.show()

In [ ]:
# Predict the classification for the whole data: use the whole TFIDF
t1 = time.time()
y_tfidf = classifier_nn.predict(tfidf)
t2 = time.time()

print(t2-t1)

In [ ]:
# The TFIDF keep the order from index. Just add the prediction as new variable
df_merge['Prediction_NN'] = y_tfidf
df_merge.head()

In [ ]:
# save classification results
df_merge.to_csv("/project/biocomplexity/sdad/projects_data/ncses/prd/Digital_abstract_labelled/big_data/tfidf_old_training_set.csv")